# Chapter 6 | Case Study A1 | **Comparing Online and Offline Prices: Testing the Difference** #

In this notebook, I'll be taking notes of the author's code on the referred case study. The goal is to reproduce the code in the original case studies repo. We shall delve into the topic of **Testing Hypothesis**. We will be using the <code>billion-prices</code> dataset.

**Main question**: are online and offline prices of same products different?
**Chosen statistic**: average of the difference of online vs offline prices.

## 1. Read the data ##

In [6]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
from mizani.formatters import percent_format
from plotnine import *

warnings.filterwarnings("ignore")

In [4]:
# Current script folder
current_path = os.getcwd()
dirname = f"{current_path}/"

# Get location folders
data_in = f"{dirname}da_data_repo/billion-prices/clean/"
data_out = f"{dirname}da_case_studies/ch06-online_offline_price_test/data/clean/"
output = f"{dirname}da_data_exercises/ch06-online_offline_price_test/output/"
func = f"{dirname}da_case_studies/ch00-tech_prep/"
sys.path.append(func)

In [5]:
from py_helper_functions import *

In [7]:
data = pd.read_csv(data_in + "online_offline_ALL_clean.csv", encoding="latin1")

In [8]:
data.head()

,COUNTRY,retailer,retailer_s,date,day,month,year,id,price,price_online,...,DEVICEID,TIME,ZIPCODE,PHOTO,OTHERSKUITEM,COMMENTS,PRICETYPE,CODE,sale_online,country_s
0,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,201209030113,429.0,429.0,...,891df49fb1b12aa0,21:03,8300,20150319_210351.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
1,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4710268235965,189.0,189.0,...,891df49fb1b12aa0,21:26,8300,20150319_212653.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
2,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524916874,6999.0,6999.0,...,891df49fb1b12aa0,21:19,8300,20150319_211929.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
3,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524925784,1999.0,2099.0,...,891df49fb1b12aa0,21:08,8300,20150319_210847.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
4,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524931310,2899.0,2899.0,...,891df49fb1b12aa0,21:06,8300,20150319_210627.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina


Let's review our scenario:
- We use data from the **United States**.
- We include products with their **regular prices** recorded.

Let's filter our data accordingly.

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45253 entries, 0 to 45252
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   COUNTRY       45253 non-null  object 
 1   retailer      45253 non-null  int64  
 2   retailer_s    45253 non-null  object 
 3   date          45253 non-null  object 
 4   day           44928 non-null  float64
 5   month         44928 non-null  float64
 6   year          44928 non-null  float64
 7   id            45253 non-null  object 
 8   price         45253 non-null  float64
 9   price_online  45253 non-null  float64
 10  imputed       22414 non-null  float64
 11  DEVICEID      45253 non-null  object 
 12  TIME          45253 non-null  object 
 13  ZIPCODE       45253 non-null  object 
 14  PHOTO         45203 non-null  object 
 15  OTHERSKUITEM  20454 non-null  object 
 16  COMMENTS      3152 non-null   object 
 17  PRICETYPE     10905 non-null  object 
 18  CODE          42233 non-nu

In [10]:
data["COUNTRY"].unique()

array(['ARGENTINA', 'AUSTRALIA', 'BRAZIL', 'CANADA', 'CHINA', 'GERMANY',
       'JAPAN', 'SOUTHAFRICA', 'UK', 'USA'], dtype=object)

In [11]:
data["PRICETYPE"].unique()

array([nan, 'Regular Price', 'Sale/Discounted Price'], dtype=object)

In [13]:
# Get USA and regular price data
filtered_data = data.loc[
    lambda x: (x["COUNTRY"] == "USA")
    & (x["PRICETYPE"] == "Regular Price")
    & (x["sale_online"].isnull())
    & (~x["price"].isnull())
    & (~x["price_online"].isnull())
]

In [14]:
filtered_data.describe()

,retailer,day,month,year,price,price_online,imputed,CODE,sale_online
count,6442.000000,6442.000000,6442.000000,6442.000000,6.442000e+03,6442.000000,4863.0,6442.000000,0.0
mean,51.351288,15.902980,5.508848,2015.562403,9.549998e+03,28.781333,1.0,144199.877057,NaN
std,5.543466,7.835587,5.037909,0.496129,7.479020e+05,56.961847,0.0,108520.291687,NaN
min,44.000000,1.000000,1.000000,2015.000000,4.800000e-01,0.480000,1.0,112190.000000,NaN
25%,46.000000,10.000000,1.000000,2015.000000,5.790000e+00,5.490000,1.0,112190.000000,NaN
50%,50.000000,15.000000,1.000000,2016.000000,1.199000e+01,11.990000,1.0,144211.000000,NaN
75%,57.000000,22.000000,11.000000,2016.000000,2.799000e+01,27.990000,1.0,144211.000000,NaN
max,62.000000,31.000000,12.000000,2016.000000,6.002113e+07,899.990000,1.0,856681.000000,NaN


In [15]:
filtered_data.loc[lambda x: x["price"] > 1000]

,COUNTRY,retailer,retailer_s,date,day,month,year,id,price,price_online,...,DEVICEID,TIME,ZIPCODE,PHOTO,OTHERSKUITEM,COMMENTS,PRICETYPE,CODE,sale_online,country_s
31898,USA,48,USA_48,2016-01-07,7.0,1.0,2016.0,721303,721303.0,29.95,...,b5f6d40d1ff6c6e3,14:49,02138,20160107_145002.jpg,721303,NaN,Regular Price,144211.0,NaN,USA
34433,USA,50,USA_50,2015-11-22,22.0,11.0,2015.0,593660,593660.0,24.98,...,559bdac983db109f,16:03,01701,20151122_160401.jpg,593660,NaN,Regular Price,112190.0,NaN,USA
42488,USA,59,USA_59,2015-11-29,29.0,11.0,2015.0,060021128,60021128.0,7.99,...,559bdac983db109f,18:55,01701,20151129_185514.jpg,060021128,NaN,Regular Price,112190.0,NaN,USA


There seems to be 3 products with unreasonable prices. Let's remove them.

In [16]:
filtered_data = filtered_data.loc[lambda x: x["price"] < 1000]

In [17]:
filtered_data.count()

COUNTRY         6439
retailer        6439
retailer_s      6439
date            6439
day             6439
month           6439
year            6439
id              6439
price           6439
price_online    6439
imputed         4862
DEVICEID        6439
TIME            6439
ZIPCODE         6439
PHOTO           6439
OTHERSKUITEM    4321
COMMENTS          30
PRICETYPE       6439
CODE            6439
sale_online        0
country_s       6439
dtype: int64

In [18]:
# Compare variables
filtered_data["diff"] = filtered_data["price_online"] - filtered_data["price"]
filtered_data["diff"].describe()

count    6439.000000
mean        0.054460
std         9.994452
min      -380.130000
25%        -0.040000
50%         0.000000
75%         0.000000
max       415.270000
Name: diff, dtype: float64

**Comments**:
- The mean difference is **-0.05USD**. 
- **Online prices** are, on average, **5 cents lower** in this dataset.
- There is a lot of **spread**: **10USD**

We can refine this view and see the distribution of observations.


In [20]:
diff = filtered_data["diff"]

# Within ±1usd
within_1usd = diff.between(-1, 1)
pct_within_1usd = within_1usd.mean() * 100

# Exactly same price (diff == 0)
pct_same = (diff == 0).mean() * 100

print(f"Within ±1USD:   {pct_within_1usd:.1f}%")
print(f"Same price: {pct_same:.1f}%")

Within ±1USD:   86.8%
Same price: 63.9%


So, roughly 87% of our products fall within the ± 1USD interval and almost 64% have the same online and offline prices.